In [38]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import pickle

In [39]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location=('/workspaces/MLOps-ZoomCamp/02-Experiment tracking and model '
 'management/mlruns/1'), creation_time=1748500545859, experiment_id='1', last_update_time=1748500545859, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

# 01.data preprocessing 

In [40]:
def read_dataframe(filename):
    if filename.endswith(".csv"):
        df = pd.read_csv(filename)

        df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
        df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

    elif filename.endswith(".parquet"):
        df = pd.read_parquet(filename)

    df["duration"] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

In [41]:
df_train = read_dataframe("../data/yellow_tripdata_2023-01.parquet")
df_val = read_dataframe("../data/yellow_tripdata_2023-02.parquet")

In [42]:
categorical = ["PULocationID", "DOLocationID"]

# Turn dataframes into list of dictionaries
train_dicts = df_train[categorical].to_dict(orient="records")
val_dicts = df_val[categorical].to_dict(orient="records")

In [43]:
# Fit dictionary vectorizer on Training data
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

# Apply learned dictionary vectorizer on validation data
X_val = dv.transform(val_dicts)

In [44]:
# Set GT Values
y_train = df_train["duration"].values
y_val = df_val["duration"].values

# 02. trying out Linear Regression without logging

In [45]:
# Train linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [46]:
# Make predictions on validation data
y_pred = lr.predict(X_val)

# Calculate RMSE on validation
rmse = root_mean_squared_error(y_val, y_pred)

In [47]:
rmse

7.811817745843695

# 03.trying out Lasso with simple logging

In [48]:
# trying and logging lasso
with mlflow.start_run():
    mlflow.set_tag("developer", "Wahba")

    mlflow.log_param("train-data-path", "../data/yellow_tripdata_2023-01.parquet")
    mlflow.log_param("valid-data-path", "../data/yellow_tripdata_2023-02.parquet")

    alpha = 0.001
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

# 04.Hyperparameter Optimization for XGBoost Using Hyperopt and MLflow

This section performs hyperparameter tuning for an XGBoost model using Hyperopt, with each trial's parameters and RMSE logged to MLflow for experiment tracking.

**The best hyperparameters identified will be used below for final model training.**


In [49]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [50]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50,
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {"loss": rmse, "status": STATUS_OK}


search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
    "seed": 42,
}
## skipping it as i already ran it for hours XD
# best_result = fmin(
#     fn=objective, space=search_space, algo=tpe.suggest, max_evals=50, trials=Trials()
# )

## Training XGBoost Model with Manual Hyperparameters and Logging to MLflow

In [ ]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    params = {
        "learning_rate": 0.37423890981602775,
        "max_depth": 80,
        "min_child_weight": 2.4679201221277838,
        "objective": "reg:squarederror",
        "reg_alpha": 0.008474358456426086,
        "reg_lambda": 0.006361637376418637,
        "seed": 42,
    }

    # autolog for xgboost (disabled it, kernel crashing in GH codesapces)
    # mlflow.xgboost.autolog(disable=True)

    # manual Log parameters
    mlflow.log_params(params)

    booster = xgb.train(
        params=params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds=50,
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    # log preprocessor (the dict-vectorizer)
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models")
